# LeetCode 937 — Reorder Data in Log Files bằng Python

Link đề: https://leetcode.com/problems/reorder-data-in-log-files/ · Độ khó: **Medium** · Chủ đề: Array, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        ...
```

Ghi chú: đáp án là **danh sách chuỗi theo một thứ tự xác định duy nhất** (digit-log giữ nguyên thứ tự tương đối; letter-log sắp theo nội dung rồi theo định danh), nên test so sánh **đúng thứ tự**, không cần sort lại. Điểm cốt lõi: phân loại log bằng **từ đầu tiên sau định danh**, rồi dùng **khoá sắp xếp** `(0, nội dung, định danh)` cho letter-log và `(1,)` cho digit-log; `sorted` của Python là **ổn định** nên digit-log tự giữ thứ tự gốc. Oracle: tách hai nhóm rồi **chọn cực tiểu lặp** (selection sort) với phép so sánh tuple tường minh.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi log là chuỗi các từ cách nhau một dấu cách, từ đầu là **định danh**; letter-log đứng trước digit-log; letter-log sắp theo nội dung (so sánh từ điển) rồi theo định danh; digit-log giữ nguyên thứ tự.
2. Nắm công cụ Python: `str.split(" ", 1)`, `str.isdigit`, `sorted(key=...)` với khoá **tuple**, tính **ổn định** của `sort`, `functools.cmp_to_key`.
3. Đi từ **tách nhóm rồi sắp xếp từng nhóm** → **một lượt `sorted` với khoá tổng hợp** → **bộ so sánh tuỳ biến `cmp_to_key`** → **sắp xếp chèn ổn định tự viết**.
4. Hiểu mấu chốt: khoá `(loại, nội dung, định danh)` với loại `0` cho letter-log và `1` cho digit-log; digit-log có khoá `(1,)` **bằng nhau** nên nhờ tính ổn định chúng giữ thứ tự ban đầu.
5. Tránh lỗi kinh điển: kiểm tra `isdigit` trên **toàn bộ nội dung** (có dấu cách nên luôn `False`), so sánh cả log (định danh trước nội dung), bỏ tiêu chí phụ (định danh) khi nội dung trùng, sắp xếp cả digit-log, và tách chuỗi bằng `split()` làm mất dấu cách giữa các từ.

## 2. Đề bài & ví dụ

Cho mảng `logs`. Mỗi log là chuỗi các từ cách nhau **đúng một dấu cách**; từ đầu tiên là **định danh**. Có hai loại log:

- **Letter-log**: mọi từ sau định danh gồm toàn chữ cái thường.
- **Digit-log**: mọi từ sau định danh gồm toàn chữ số.

Sắp xếp lại các log sao cho:

1. Mọi letter-log đứng **trước** mọi digit-log.
2. Letter-log được sắp theo **nội dung** (phần sau định danh) theo thứ tự từ điển; nếu nội dung bằng nhau thì theo **định danh** (từ điển).
3. Digit-log **giữ nguyên** thứ tự tương đối.

Trả về mảng log sau khi sắp.

**Ví dụ 1:** `["dig1 8 1 5 1","let1 art can","dig2 3 6","let2 own kit dig","let3 art zero"]` → `["let1 art can","let3 art zero","let2 own kit dig","dig1 8 1 5 1","dig2 3 6"]`.

**Ví dụ 2:** `["a1 9 2 3 1","g1 act car","zo4 4 7","ab1 off key dog","a8 act zoo"]` → `["g1 act car","a8 act zoo","ab1 off key dog","a1 9 2 3 1","zo4 4 7"]`.

**Ràng buộc:** `1 <= logs.length <= 100`; mỗi log dài `3..100`; các từ cách nhau đúng một dấu cách; mỗi log có định danh và ít nhất một từ sau đó.

In [1]:
import itertools
import random
import time
from functools import cmp_to_key
from typing import List

examples = [
    (["dig1 8 1 5 1", "let1 art can", "dig2 3 6", "let2 own kit dig", "let3 art zero"],
     ["let1 art can", "let3 art zero", "let2 own kit dig", "dig1 8 1 5 1", "dig2 3 6"]),
    (["a1 9 2 3 1", "g1 act car", "zo4 4 7", "ab1 off key dog", "a8 act zoo"],
     ["g1 act car", "a8 act zoo", "ab1 off key dog", "a1 9 2 3 1", "zo4 4 7"]),
]
for logs, want in examples:
    print(logs, "->", want)

['dig1 8 1 5 1', 'let1 art can', 'dig2 3 6', 'let2 own kit dig', 'let3 art zero'] -> ['let1 art can', 'let3 art zero', 'let2 own kit dig', 'dig1 8 1 5 1', 'dig2 3 6']
['a1 9 2 3 1', 'g1 act car', 'zo4 4 7', 'ab1 off key dog', 'a8 act zoo'] -> ['g1 act car', 'a8 act zoo', 'ab1 off key dog', 'a1 9 2 3 1', 'zo4 4 7']


## 3. Kiến thức nền

### 3.1 Tách định danh và nội dung

`log.split(" ", 1)` tách **một lần** tại dấu cách đầu tiên: `["let1", "art can"]`. Nội dung giữ nguyên các dấu cách bên trong (khác `split()` không tham số).

In [2]:
print("let1 art can".split(" ", 1), "| split() làm mất cấu trúc:", "let1 art can".split())
ident, content = "dig1 8 1 5 1".split(" ", 1)
print(repr(ident), repr(content), "| isdigit() của cả nội dung:", content.isdigit(), "| của từ đầu:", content.split(" ")[0].isdigit())

['let1', 'art can'] | split() làm mất cấu trúc: ['let1', 'art', 'can']
'dig1' '8 1 5 1' | isdigit() của cả nội dung: False | của từ đầu: True


### 3.2 Phân loại: dùng từ đầu tiên sau định danh

Đề bảo đảm **mọi** từ sau định danh cùng loại, nên chỉ cần xét **ký tự đầu của nội dung** (hoặc từ đầu): chữ số ⇒ digit-log, ngược lại ⇒ letter-log. Không dùng `content.isdigit()` vì nội dung có dấu cách (`"8 1 5 1".isdigit()` là `False`).

### 3.3 Khoá sắp xếp tuple và tính ổn định

`sorted(xs, key=f)` so sánh các khoá; tuple so sánh **từng phần tử từ trái sang phải**. `sort` của Python **ổn định**: các phần tử có khoá **bằng nhau** giữ nguyên thứ tự ban đầu. Đặt khoá digit-log là hằng `(1,)` để chúng "bằng nhau" và giữ thứ tự gốc.

### 3.4 So sánh chuỗi từ điển

`"art can" < "art zero"` vì tại vị trí khác nhau đầu tiên `'c' < 'z'`; so sánh bằng mã ký tự. Chuỗi là **tiền tố** của chuỗi kia thì ngắn hơn nhỏ hơn: `"art" < "art can"`.

In [3]:
key = lambda log: (lambda ident, content: (0, content, ident) if not content[0].isdigit() else (1,))(*log.split(" ", 1))
print([key(l) for l in examples[0][0]])
print(sorted(examples[0][0], key=key) == examples[0][1], "| 'art' < 'art can':", "art" < "art can", "| 'a b' < 'ab':", "a b" < "ab")

[(1,), (0, 'art can', 'let1'), (1,), (0, 'own kit dig', 'let2'), (0, 'art zero', 'let3')]
True | 'art' < 'art can': True | 'a b' < 'ab': True


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1. Phân loại: `dig1`, `dig2` là digit-log; `let1`, `let2`, `let3` là letter-log.

- Letter-log theo nội dung: `"art can"` (let1), `"art zero"` (let3), `"own kit dig"` (let2) ⇒ thứ tự `let1, let3, let2`.
- Digit-log giữ thứ tự gốc: `dig1`, `dig2`.
- Ghép: letter-log rồi digit-log.

Khi **nội dung trùng nhau**, ví dụ `"g1 act car"` và `"a8 act car"`: theo định danh `a8 < g1` nên `a8` đứng trước.

In [4]:
logs = ["g1 act car", "a8 act car", "z0 1 2", "b2 act", "a1 3"]
letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
print("letter:", letters, "\ndigit :", digits)
print("sắp letter theo (nội dung, định danh):", sorted(letters, key=lambda l: (l.split(" ", 1)[1], l.split(" ", 1)[0])))

letter: ['g1 act car', 'a8 act car', 'b2 act'] 
digit : ['z0 1 2', 'a1 3']
sắp letter theo (nội dung, định danh): ['b2 act', 'a8 act car', 'g1 act car']


## 5. Cách 1 — Tách nhóm rồi sắp xếp từng nhóm

**Ý tưởng:** duyệt `logs`, đưa vào hai danh sách `letters` và `digits` (giữ thứ tự gặp). Sắp `letters` theo `(nội dung, định danh)`; trả `letters + digits`.

In [5]:
def reorder_split(logs):
    letters, digits = [], []
    for log in logs:
        ident, content = log.split(" ", 1)
        (digits if content[0].isdigit() else letters).append(log)    # phân loại theo ký tự đầu của nội dung
    letters.sort(key=lambda log: (log.split(" ", 1)[1], log.split(" ", 1)[0]))     # (nội dung, định danh)
    return letters + digits                                           # digit-log giữ nguyên thứ tự gốc


for logs, want in examples:
    assert reorder_split(logs) == want
print([reorder_split(l) == w for l, w in examples])

[True, True]


In [6]:
# Code nộp LeetCode cho Cách 1
class SolutionSplit:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        letters, digits = [], []
        for log in logs:
            ident, content = log.split(" ", 1)
            (digits if content[0].isdigit() else letters).append(log)
        letters.sort(key=lambda log: (log.split(" ", 1)[1], log.split(" ", 1)[0]))
        return letters + digits

**Độ phức tạp:** `O(n log n · L)` thời gian (so sánh chuỗi dài `L`), `O(n · L)` bộ nhớ.

**Nút thắt:** phải duy trì hai danh sách và ghép; một lượt `sorted` với khoá thích hợp làm cả hai việc.

## 6. Ý tưởng thuật toán / bước đột phá

Một khoá tổng hợp: `(0, nội dung, định danh)` cho letter-log (đứng trước vì `0 < 1`), `(1,)` cho digit-log. `sorted` ổn định giữ nguyên thứ tự digit-log vì mọi khoá của chúng **bằng nhau**.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** một lượt `sorted` với khoá tuple, **Cách 3** bộ so sánh `cmp_to_key`, **Cách 4** sắp xếp chèn ổn định tự viết.

### Cách 2 — Một lượt `sorted` với khoá tuple

**Giải thích từng đoạn:**

1. `key(log)`: tách `ident, content = log.split(" ", 1)`.
2. Letter-log: `(0, content, ident)`; digit-log: `(1,)`.
3. `sorted(logs, key=key)`: ổn định nên digit-log giữ thứ tự.

In [7]:
def reorder_key(logs):
    def key(log):
        ident, content = log.split(" ", 1)
        return (1,) if content[0].isdigit() else (0, content, ident)   # digit-log: khoá hằng; letter-log: (loại, nội dung, định danh)

    return sorted(logs, key=key)


for logs, want in examples:
    assert reorder_key(logs) == want
print([reorder_key(l) == w for l, w in examples])

[True, True]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionKey:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        def key(log):
            ident, content = log.split(" ", 1)
            return (1,) if content[0].isdigit() else (0, content, ident)

        return sorted(logs, key=key)

**Độ phức tạp:** `O(n log n · L)` thời gian, `O(n · L)` bộ nhớ. Ngắn gọn nhất.

### Cách 3 — Bộ so sánh tuỳ biến `cmp_to_key`

**Ý tưởng:** viết hàm `cmp(a, b)` trả số âm/`0`/dương theo luật đề; chuyển bằng `cmp_to_key`. Luật: cả hai digit ⇒ `0` (giữ thứ tự nhờ ổn định); a letter, b digit ⇒ `-1`; a digit, b letter ⇒ `1`; cả hai letter ⇒ so `(nội dung, định danh)`.

In [9]:
def reorder_cmp(logs):
    def split(log):
        ident, content = log.split(" ", 1)
        return ident, content, content[0].isdigit()

    def cmp(a, b):
        ia, ca, da = split(a)
        ib, cb, db = split(b)
        if da and db:
            return 0                                     # hai digit-log: coi như bằng nhau (sort ổn định giữ thứ tự)
        if da:
            return 1                                     # digit-log sau letter-log
        if db:
            return -1
        if ca != cb:
            return -1 if ca < cb else 1                  # so nội dung
        if ia != ib:
            return -1 if ia < ib else 1                  # nội dung bằng nhau: so định danh
        return 0

    return sorted(logs, key=cmp_to_key(cmp))


for logs, want in examples:
    assert reorder_cmp(logs) == want
print([reorder_cmp(l) == w for l, w in examples])

[True, True]


In [10]:
# Code nộp LeetCode cho Cách 3
from functools import cmp_to_key


class SolutionCmp:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        def split(log):
            ident, content = log.split(" ", 1)
            return ident, content, content[0].isdigit()

        def cmp(a, b):
            ia, ca, da = split(a)
            ib, cb, db = split(b)
            if da and db:
                return 0
            if da:
                return 1
            if db:
                return -1
            if ca != cb:
                return -1 if ca < cb else 1
            if ia != ib:
                return -1 if ia < ib else 1
            return 0

        return sorted(logs, key=cmp_to_key(cmp))

**Độ phức tạp:** `O(n log n)` lần so sánh, mỗi lần `O(L)`. Linh hoạt hơn khoá tuple khi luật sắp xếp phức tạp.

### Cách 4 — Sắp xếp chèn ổn định tự viết

**Ý tưởng:** không dùng `sort` có sẵn. Duyệt từng log, **chèn** vào danh sách kết quả tại vị trí sau mọi phần tử "không lớn hơn" nó (tìm bằng so sánh tuple tường minh). Chèn **sau** các phần tử bằng nhau ⇒ ổn định ⇒ digit-log giữ thứ tự.

In [11]:
def reorder_insertion(logs):
    def key(log):
        ident, content = log.split(" ", 1)
        return (1,) if content[0].isdigit() else (0, content, ident)

    out = []
    for log in logs:
        k = key(log)
        pos = len(out)
        while pos > 0 and key(out[pos - 1]) > k:         # lùi qua các phần tử LỚN HƠN HẲN: dừng sau phần tử bằng nhau (ổn định)
            pos -= 1
        out.insert(pos, log)
    return out


for logs, want in examples:
    assert reorder_insertion(logs) == want
print([reorder_insertion(l) == w for l, w in examples])

[True, True]


In [12]:
# Code nộp LeetCode cho Cách 4 (O(n^2): n <= 100 nên vẫn nhanh)
class SolutionInsertion:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        def key(log):
            ident, content = log.split(" ", 1)
            return (1,) if content[0].isdigit() else (0, content, ident)

        out = []
        for log in logs:
            k, pos = key(log), len(out)
            while pos > 0 and key(out[pos - 1]) > k:
                pos -= 1
            out.insert(pos, log)
        return out

**Độ phức tạp:** `O(n²)` so sánh trong trường hợp xấu (n ≤ 100 ⇒ tối đa `~5000`); bộ nhớ `O(n)`.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 1:

| log | `ident` | `content` | loại | khoá |
|---|---|---|---|---|
| `dig1 8 1 5 1` | `dig1` | `8 1 5 1` | digit | `(1,)` |
| `let1 art can` | `let1` | `art can` | letter | `(0, 'art can', 'let1')` |
| `dig2 3 6` | `dig2` | `3 6` | digit | `(1,)` |
| `let2 own kit dig` | `let2` | `own kit dig` | letter | `(0, 'own kit dig', 'let2')` |
| `let3 art zero` | `let3` | `art zero` | letter | `(0, 'art zero', 'let3')` |

Sắp tăng theo khoá: `(0,'art can',…) < (0,'art zero',…) < (0,'own kit dig',…) < (1,) = (1,)`; hai khoá `(1,)` bằng nhau nên giữ thứ tự `dig1`, `dig2` ✓. Hàm trace:

In [13]:
def trace_key(logs):
    rows = []
    for idx, log in enumerate(logs):
        ident, content = log.split(" ", 1)
        k = (1,) if content[0].isdigit() else (0, content, ident)
        rows.append((idx, log, k))
        print(f"{idx}: {log!r:22} -> khoá {k}")
    out = [log for _, log, _ in sorted(rows, key=lambda r: r[2])]
    print("sau sắp xếp ổn định:", out)
    return out


assert trace_key(examples[0][0]) == examples[0][1]

0: 'dig1 8 1 5 1'         -> khoá (1,)
1: 'let1 art can'         -> khoá (0, 'art can', 'let1')
2: 'dig2 3 6'             -> khoá (1,)
3: 'let2 own kit dig'     -> khoá (0, 'own kit dig', 'let2')
4: 'let3 art zero'        -> khoá (0, 'art zero', 'let3')
sau sắp xếp ổn định: ['let1 art can', 'let3 art zero', 'let2 own kit dig', 'dig1 8 1 5 1', 'dig2 3 6']


**Trace Cách 4** (chèn) với ví dụ 2: `a1 9 2 3 1` (khoá `(1,)`) vào rỗng ⇒ `[a1…]`; `g1 act car` khoá `(0,...)` nhỏ hơn `(1,)` ⇒ lùi qua `a1`, chèn đầu; `zo4 4 7` `(1,)` bằng `a1`'s khoá ⇒ không lùi (ổn định), chèn cuối; `ab1 off key dog` và `a8 act zoo` chèn vào vùng letter theo nội dung.

## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt `sorted` với khoá tuple). Giữ nguyên `class Solution`, `reorderLogFiles(self, logs)`:

In [14]:
class Solution:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        def key(log: str):
            ident, content = log.split(" ", 1)                  # tách một lần: giữ nguyên dấu cách trong nội dung
            if content[0].isdigit():
                return (1,)                                    # digit-log: khoá bằng nhau -> giữ thứ tự (sort ổn định)
            return (0, content, ident)                         # letter-log: trước digit-log, rồi nội dung, rồi định danh

        return sorted(logs, key=key)


for logs, want in examples:
    assert Solution().reorderLogFiles(list(logs)) == want
print([Solution().reorderLogFiles(list(l)) == w for l, w in examples])

[True, True]


## 10. Vì sao đúng

**(a) Phân loại.** Đề bảo đảm mọi từ sau định danh cùng loại; do đó ký tự đầu của nội dung (chữ cái hoặc chữ số) xác định loại. `content[0].isdigit()` ⇔ digit-log.

**(b) Thứ tự mong muốn là thứ tự tổng.** Định nghĩa quan hệ `a ≺ b`: (i) letter ≺ digit; (ii) cả hai letter: `(nội dung_a, định danh_a) < (nội dung_b, định danh_b)` (từ điển, lần lượt); (iii) cả hai digit: không phân biệt (giữ thứ tự gốc). Quan hệ này là **thứ tự yếu** (preorder tổng): letter-log được xếp theo khoá `(nội dung, định danh)` hoàn toàn, digit-log bằng nhau đôi một và đứng sau.

**(c) Khoá tuple biểu diễn đúng thứ tự.** So sánh tuple là từ điển theo từng phần tử. `(0, c, i) < (1,)` vì `0 < 1`; hai tuple `(0, c₁, i₁)`, `(0, c₂, i₂)` so lần lượt `c` rồi `i`; hai tuple `(1,)` bằng nhau. Do đó thứ tự của khoá đúng bằng `≺`.

**(d) Tính ổn định giữ thứ tự digit.** Thuật toán sắp xếp **ổn định** giữ nguyên thứ tự tương đối của các phần tử có khoá bằng nhau. Mọi digit-log có khoá `(1,)` nên thứ tự tương đối ban đầu được giữ. (Hai letter-log có cả nội dung và định danh bằng nhau là hai chuỗi **giống hệt** nhau nên thứ tự không phân biệt được.)

**(e) Cách 1** tách nhóm (giữ thứ tự gặp ⇒ digit-log giữ thứ tự), sắp letter-log theo khoá. **Cách 3**: `cmp` tương ứng với `≺`, trả `0` cho hai digit-log nên `sorted` ổn định giữ thứ tự. **Cách 4**: chèn **sau** mọi phần tử có khoá `≤` (dừng khi gặp phần tử `≤`) nên là sắp chèn ổn định.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **chọn cực tiểu lặp** (selection sort tường minh) trên mọi dãy con có thứ tự của một bộ log mẫu (mục 12D) và trên log ngẫu nhiên (12E).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Tách nhóm + sort letter | `O(n log n · L)` | `O(n·L)` | trực quan |
| 2. `sorted` với khoá tuple | `O(n log n · L)` | `O(n·L)` | **đáp án chính** |
| 3. `cmp_to_key` | `O(n log n · L)` | `O(n·L)` | linh hoạt |
| 4. Sắp chèn ổn định | `O(n² · L)` | `O(n·L)` | tự viết, `n ≤ 100` |

Đo thực tế (`n = 100` và `n = 5000` để thấy khác biệt):

In [15]:
def rand_log(rng, kind=None, vocab=("art", "can", "zero", "own", "kit", "dig", "a", "ab", "b"), ids=("a1", "a8", "g1", "let1", "let2", "z0", "ab1")):
    kind = kind or rng.choice(["letter", "digit"])
    ident = rng.choice(ids)
    if kind == "letter":
        words = [rng.choice(vocab) for _ in range(rng.randint(1, 4))]
    else:
        words = [str(rng.randint(0, 99)) for _ in range(rng.randint(1, 4))]
    return ident + " " + " ".join(words)


def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(937)
for n in (100, 5000):
    logs = [rand_log(rng) for _ in range(n)]
    row = [f"{name} {bench(fn, logs, 1 if name == 'insertion' and n > 1000 else 3) * 1e3:8.2f} ms" for name, fn in
           [("split", reorder_split), ("key", reorder_key), ("cmp", reorder_cmp), ("insertion", reorder_insertion)]]
    print(f"n = {n:5d}: " + " | ".join(row))

n =   100: split     0.03 ms | key     0.02 ms | cmp     0.18 ms | insertion     0.38 ms


n =  5000: split     2.27 ms | key     1.92 ms | cmp    16.54 ms | insertion  1060.62 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi dãy con có thứ tự của bộ log mẫu, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn.

### A. Ví dụ của đề

In [16]:
def oracle(logs):
    """Oracle: tách hai nhóm; letter-log được chọn LẶP cực tiểu (selection sort) theo (nội dung, định danh) bằng so sánh tuple tường minh;
    digit-log giữ thứ tự gốc."""
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    out = []
    pool = letters[:]
    while pool:
        best = 0
        for i in range(1, len(pool)):
            ci, ii = pool[i].split(" ", 1)[1], pool[i].split(" ", 1)[0]
            cb, ib = pool[best].split(" ", 1)[1], pool[best].split(" ", 1)[0]
            if (ci, ii) < (cb, ib):
                best = i
        out.append(pool.pop(best))
    return out + digits


IMPLS = {
    "split": reorder_split,
    "key": reorder_key,
    "cmp": reorder_cmp,
    "insertion": reorder_insertion,
    "Solution": lambda l: Solution().reorderLogFiles(list(l)),
    "SolutionSplit": lambda l: SolutionSplit().reorderLogFiles(list(l)),
    "SolutionKey": lambda l: SolutionKey().reorderLogFiles(list(l)),
    "SolutionCmp": lambda l: SolutionCmp().reorderLogFiles(list(l)),
    "SolutionInsertion": lambda l: SolutionInsertion().reorderLogFiles(list(l)),
}


def check(logs, want=None, names=None):
    if want is None:
        want = oracle(logs)
    for name in (names or IMPLS):
        got = IMPLS[name](list(logs))
        assert got == want, (name, logs, got, want)


for logs, want in examples:
    assert oracle(logs) == want
    check(logs, want)
print("ví dụ đề: oracle chọn-cực-tiểu và", len(IMPLS), "cài đặt đúng (so sánh ĐÚNG THỨ TỰ)")

ví dụ đề: oracle chọn-cực-tiểu và 9 cài đặt đúng (so sánh ĐÚNG THỨ TỰ)


### B. Edge case

- Một log (letter hoặc digit).
- Toàn digit-log: giữ nguyên thứ tự gốc.
- Toàn letter-log.
- Nội dung trùng nhau, định danh khác nhau (sắp theo định danh).
- Nội dung là **tiền tố** của nội dung khác (`"art"` và `"art can"`).
- Log trùng hoàn toàn (hai bản giống hệt).
- Định danh chứa chữ số/chữ cái hỗn hợp (`a1`, `a10`, `a2`: so từ điển, `a10 < a2`).
- Letter-log xen kẽ digit-log.

In [17]:
cases = [
    (["a1 b"], ["a1 b"]),
    (["a1 1"], ["a1 1"]),
    (["z 3 2", "a 1", "m 9"], ["z 3 2", "a 1", "m 9"]),
    (["q b", "a b", "z a"], ["z a", "a b", "q b"]),
    (["g1 act car", "a8 act car", "b2 act car"], ["a8 act car", "b2 act car", "g1 act car"]),
    (["x1 art can", "x2 art"], ["x2 art", "x1 art can"]),
    (["a1 abc", "a1 abc"], ["a1 abc", "a1 abc"]),
    (["a2 x", "a10 x", "a1 x"], ["a1 x", "a10 x", "a2 x"]),
    (["d1 5", "l1 zzz", "d2 6", "l2 aaa"], ["l2 aaa", "l1 zzz", "d1 5", "d2 6"]),
    (["id a b", "id ab"], ["id a b", "id ab"]),
]
for logs, want in cases:
    check(logs, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

10 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_content_only(logs):                            # sắp letter-log chỉ theo nội dung: nội dung trùng giữ thứ tự gốc
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    return sorted(letters, key=lambda l: l.split(" ", 1)[1]) + digits


def bug_whole_log(logs):                               # sắp theo cả chuỗi log (định danh đứng trước nội dung)
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    return sorted(letters) + digits


def bug_sort_digits(logs):                             # sắp cả digit-log
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    return sorted(letters, key=lambda l: (l.split(" ", 1)[1], l.split(" ", 1)[0])) + sorted(digits)


def bug_isdigit_whole(logs):                           # content.isdigit() trên cả nội dung (luôn False khi có dấu cách)
    key = lambda l: (1,) if l.split(" ", 1)[1].isdigit() else (0, l.split(" ", 1)[1], l.split(" ", 1)[0])
    return sorted(logs, key=key)


def bug_split_all(logs):                               # split() không tham số rồi nối lại bằng '' (mất dấu cách)
    def key(l):
        parts = l.split()
        content = "".join(parts[1:])
        return (1,) if content[0].isdigit() else (0, content, parts[0])

    return sorted(logs, key=key)


def bug_digits_first(logs):                            # digit-log đứng trước letter-log
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    return digits + sorted(letters, key=lambda l: (l.split(" ", 1)[1], l.split(" ", 1)[0]))


BUGS = [bug_content_only, bug_whole_log, bug_sort_digits, bug_isdigit_whole, bug_split_all, bug_digits_first]
rng = random.Random(3)
tests = [[rand_log(rng) for _ in range(rng.randint(1, 8))] for _ in range(1500)]
for bug in BUGS:
    bad = next((l for l in tests if bug(l) != oracle(l)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:18s} sai trên {bad}:\n    ra {bug(bad)}\n    đúng {oracle(bad)}")

bug_content_only   sai trên ['g1 ab', 'let2 own can a zero', 'z0 can zero ab', 'a1 14 30 69 53', 'a1 ab', 'ab1 own dig', 'let1 37 9 13 45']:
    ra ['g1 ab', 'a1 ab', 'z0 can zero ab', 'let2 own can a zero', 'ab1 own dig', 'a1 14 30 69 53', 'let1 37 9 13 45']
    đúng ['a1 ab', 'g1 ab', 'z0 can zero ab', 'let2 own can a zero', 'ab1 own dig', 'a1 14 30 69 53', 'let1 37 9 13 45']
bug_whole_log      sai trên ['z0 50 93 73 56', 'g1 art', 'let1 kit a', 'let1 73 44 68 74', 'let2 43 87', 'ab1 zero dig b', 'z0 kit kit', 'a1 ab can dig can']:
    ra ['a1 ab can dig can', 'ab1 zero dig b', 'g1 art', 'let1 kit a', 'z0 kit kit', 'z0 50 93 73 56', 'let1 73 44 68 74', 'let2 43 87']
    đúng ['a1 ab can dig can', 'g1 art', 'let1 kit a', 'z0 kit kit', 'ab1 zero dig b', 'z0 50 93 73 56', 'let1 73 44 68 74', 'let2 43 87']
bug_sort_digits    sai trên ['z0 50 93 73 56', 'g1 art', 'let1 kit a', 'let1 73 44 68 74', 'let2 43 87', 'ab1 zero dig b', 'z0 kit kit', 'a1 ab can dig can']:
    ra ['a1 ab can dig ca

### D. Vét cạn mọi dãy con có thứ tự của bộ log mẫu

Bộ log mẫu (6 log, có nội dung trùng, nội dung tiền tố, định danh trùng, 2 digit-log): `["g1 act car", "a8 act car", "b2 act", "d1 3 6", "d2 1 9", "a1 act car"]`. Xét **mọi dãy có thứ tự** gồm `k` log phân biệt-vị-trí (`k = 1..6`): `6 + 30 + 120 + 360 + 720 + 720 = 1 956` dãy.

In [19]:
sample = ["g1 act car", "a8 act car", "b2 act", "d1 3 6", "d2 1 9", "a1 act car"]
t0 = time.perf_counter()
cnt = 0
for k in range(1, 7):
    for perm in itertools.permutations(sample, k):
        logs = list(perm)
        want = oracle(logs)
        for name in IMPLS:
            assert IMPLS[name](logs) == want, (name, logs)
        cnt += 1
print(cnt, "dãy log có thứ tự: 9 cài đặt khớp oracle chọn-cực-tiểu;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 1956

1956 dãy log có thứ tự: 9 cài đặt khớp oracle chọn-cực-tiểu; 0.1 s


### E. Ngẫu nhiên có seed và tính chất

Log ngẫu nhiên tới `100` log (từ vựng nhỏ để có nhiều trùng nội dung/định danh). Tính chất: (1) đầu ra là **hoán vị** của đầu vào (cùng đa tập log); (2) letter-log đứng trước digit-log; (3) digit-log trong đầu ra giữ **thứ tự tương đối** như đầu vào; (4) **bất biến theo thứ tự đầu vào của letter-log** (xáo trộn letter-log trong đầu vào không đổi phần letter của đầu ra, vì khoá toàn phần); (5) idempotent: sắp lại đầu ra không đổi.

In [20]:
rng = random.Random(20240937)
for _ in range(1500):
    n = rng.randint(1, 100)
    logs = [rand_log(rng) for _ in range(n)]
    want = oracle(logs) if n <= 40 else reorder_key(logs)
    for name in ("split", "key", "cmp", "insertion", "Solution", "SolutionSplit", "SolutionKey", "SolutionCmp", "SolutionInsertion"):
        assert IMPLS[name](logs) == want, (name, logs)
    out = Solution().reorderLogFiles(list(logs))
    assert sorted(out) == sorted(logs)                                              # (1)
    is_digit = lambda l: l.split(" ", 1)[1][0].isdigit()
    kinds = [is_digit(l) for l in out]
    assert kinds == sorted(kinds)                                                   # (2) False... rồi True...
    assert [l for l in out if is_digit(l)] == [l for l in logs if is_digit(l)]      # (3)
    letters_in = [l for l in logs if not is_digit(l)]
    rng.shuffle(letters_in)
    it = iter(letters_in)
    shuffled = [next(it) if not is_digit(l) else l for l in logs]
    assert [l for l in Solution().reorderLogFiles(shuffled) if not is_digit(l)] == [l for l in out if not is_digit(l)]   # (4)
    assert Solution().reorderLogFiles(list(out)) == out                             # (5)
print("1500 dãy log ngẫu nhiên: 9 cài đặt khớp; 5 tính chất (hoán vị, loại, giữ thứ tự digit, bất biến thứ tự letter, idempotent) đúng")

1500 dãy log ngẫu nhiên: 9 cài đặt khớp; 5 tính chất (hoán vị, loại, giữ thứ tự digit, bất biến thứ tự letter, idempotent) đúng


### F. Giới hạn

`n = 100` log dài `100` ký tự (nội dung dài, nhiều tiền tố chung), và `n = 5000` log: các cách nhanh khớp nhau.

In [21]:
rng = random.Random(500)
long_logs = []
for _ in range(100):
    ident = rng.choice(["id1", "id2", "id3"])
    body = " ".join(rng.choice(["alpha", "beta", "gamma"]) for _ in range(12))
    long_logs.append((ident + " " + body)[:100].rstrip())
for _ in range(20):
    long_logs[rng.randrange(100)] = "d" + str(rng.randint(0, 9)) + " " + " ".join(str(rng.randint(0, 99)) for _ in range(30))[:90].rstrip()
assert all(3 <= len(l) <= 100 for l in long_logs)
res = {k: IMPLS[k](long_logs) for k in IMPLS}
assert len({tuple(v) for v in res.values()}) == 1 and res["key"] == oracle(long_logs)
big = [rand_log(rng) for _ in range(5000)]
fast = ("split", "key", "cmp", "Solution", "SolutionSplit", "SolutionKey", "SolutionCmp")
assert len({tuple(IMPLS[k](big)) for k in fast}) == 1
print("n = 100 (log dài 100 ký tự): 9 cài đặt khớp oracle; n = 5000: 7 cài đặt nhanh khớp nhau")

n = 100 (log dài 100 ký tự): 9 cài đặt khớp oracle; n = 5000: 7 cài đặt nhanh khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sắp letter-log chỉ theo nội dung

In [22]:
logs = ["g1 act car", "a8 act car"]
print("chỉ nội dung:", bug_content_only(logs), "| đúng:", Solution().reorderLogFiles(logs))
assert bug_content_only(logs) == ["g1 act car", "a8 act car"] and Solution().reorderLogFiles(logs) == ["a8 act car", "g1 act car"]

chỉ nội dung: ['g1 act car', 'a8 act car'] | đúng: ['a8 act car', 'g1 act car']


**Vì sao sai:** khi nội dung trùng phải so định danh. Chỉ theo nội dung thì `sorted` ổn định giữ thứ tự gốc. **Đúng:** khoá `(nội dung, định danh)`.

### Lỗi 2: sắp theo cả chuỗi log

In [23]:
logs = ["a1 zzz", "b1 aaa"]
print("cả chuỗi:", bug_whole_log(logs), "| đúng:", Solution().reorderLogFiles(logs))
assert bug_whole_log(logs) == ["a1 zzz", "b1 aaa"] and Solution().reorderLogFiles(logs) == ["b1 aaa", "a1 zzz"]

cả chuỗi: ['a1 zzz', 'b1 aaa'] | đúng: ['b1 aaa', 'a1 zzz']


**Vì sao sai:** chuỗi log bắt đầu bằng **định danh** nên so sánh cả chuỗi sắp theo định danh trước. **Đúng:** nội dung trước.

### Lỗi 3: sắp cả digit-log

In [24]:
logs = ["d2 5", "d1 9"]
print("sắp digit:", bug_sort_digits(logs), "| đúng:", Solution().reorderLogFiles(logs))
assert bug_sort_digits(logs) == ["d1 9", "d2 5"] and Solution().reorderLogFiles(logs) == ["d2 5", "d1 9"]

sắp digit: ['d1 9', 'd2 5'] | đúng: ['d2 5', 'd1 9']


**Vì sao sai:** digit-log phải **giữ nguyên thứ tự**. **Đúng:** khoá hằng `(1,)` (hoặc không sắp nhóm này).

### Lỗi 4: `isdigit()` trên cả nội dung

In [25]:
logs = ["d1 8 1 5 1", "l1 art can"]
print("isdigit toàn nội dung:", bug_isdigit_whole(logs), "| đúng:", Solution().reorderLogFiles(logs))
print("'8 1 5 1'.isdigit() =", "8 1 5 1".isdigit())
assert bug_isdigit_whole(logs) != Solution().reorderLogFiles(logs)

isdigit toàn nội dung: ['d1 8 1 5 1', 'l1 art can'] | đúng: ['l1 art can', 'd1 8 1 5 1']
'8 1 5 1'.isdigit() = False


**Vì sao sai:** nội dung có dấu cách nên `isdigit()` trả `False`: digit-log bị coi là letter-log. **Đúng:** xét ký tự đầu (`content[0].isdigit()`).

### Lỗi 5: `split()` làm mất dấu cách

In [26]:
logs = ["a2 a z", "a1 ab"]
print("split + nối:", bug_split_all(logs), "| đúng:", Solution().reorderLogFiles(logs))
assert bug_split_all(logs) == ["a1 ab", "a2 a z"] and Solution().reorderLogFiles(logs) == ["a2 a z", "a1 ab"]
bad = next(l for l in tests if bug_split_all(l) != oracle(l))
print("phản ví dụ:", bad, "->", bug_split_all(bad), "| đúng:", oracle(bad))

split + nối: ['a1 ab', 'a2 a z'] | đúng: ['a2 a z', 'a1 ab']
phản ví dụ: ['a8 can can can', 'z0 a a', 'let2 a own', 'let2 own kit', 'ab1 3 56 52', 'g1 81 63 67', 'ab1 3 76 24 93', 'a1 ab zero'] -> ['z0 a a', 'a1 ab zero', 'let2 a own', 'a8 can can can', 'let2 own kit', 'ab1 3 56 52', 'g1 81 63 67', 'ab1 3 76 24 93'] | đúng: ['z0 a a', 'let2 a own', 'a1 ab zero', 'a8 can can can', 'let2 own kit', 'ab1 3 56 52', 'g1 81 63 67', 'ab1 3 76 24 93']


**Vì sao sai:** nối bỏ dấu cách làm `"a b"` thành `"ab"`: thứ tự so sánh thay đổi (`' '` nhỏ hơn mọi chữ cái). **Đúng:** `split(" ", 1)` giữ nguyên nội dung.

### Lỗi 6 (bonus): digit-log đứng trước

In [27]:
logs = ["d1 3", "l1 abc"]
print("digit trước:", bug_digits_first(logs), "| đúng:", Solution().reorderLogFiles(logs))
assert bug_digits_first(logs) == ["d1 3", "l1 abc"] and Solution().reorderLogFiles(logs) == ["l1 abc", "d1 3"]

digit trước: ['d1 3', 'l1 abc'] | đúng: ['l1 abc', 'd1 3']


**Vì sao sai:** đề: letter-log **trước** digit-log. **Đúng:** khoá loại `0` cho letter, `1` cho digit.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "sắp xếp theo nhiều tiêu chí, một số phần tử giữ nguyên thứ tự" → **khoá tuple** + **tính ổn định** của `sorted`; tiêu chí "nhóm" đặt làm phần tử đầu của tuple.

**Mẹo:**

1. Tách bằng `split(" ", 1)` khi cần giữ nguyên phần còn lại.
2. Phân loại bằng ký tự/từ đầu, không dùng thao tác trên cả chuỗi có dấu cách.
3. Phần tử cần "giữ thứ tự" → cho khoá **bằng nhau**.
4. Tuple so sánh từ trái sang phải: tiêu chí quan trọng nhất đứng trước.
5. `cmp_to_key` khi luật không biểu diễn được bằng khoá.
6. Oracle: chọn cực tiểu lặp (selection sort) tường minh.

**Họ bài cùng pattern:** Sort Characters By Frequency (451), Largest Number (179), Relative Sort Array (1122), Custom Sort String (791), Sort the People (2418), Merge Intervals (56), Sort Array by Increasing Frequency (1636).

## 15. Biến thể

1. **Giữ digit-log nhưng đặt trước letter-log.**
2. **Sắp letter-log giảm dần** (nội dung giảm, định danh tăng).
3. **Phân loại theo từ cuối** thay vì từ đầu.
4. **Sắp digit-log theo tổng chữ số** (tiêu chí mới, ổn định).
5. **Trả về chỉ số hoán vị** (argsort): vị trí gốc của từng log trong đầu ra.

Cài biến thể 2, 4, 5 và đối chiếu oracle tự viết:

In [28]:
def reorder_desc(logs):
    """Biến thể 2: letter-log nội dung GIẢM dần, trùng nội dung thì định danh TĂNG dần; digit-log giữ thứ tự."""
    def key(log):
        ident, content = log.split(" ", 1)
        if content[0].isdigit():
            return (1, 0, "", "")
        neg = tuple(-ord(c) for c in content) + (0,)        # đảo thứ tự từ điển của nội dung (kể cả tiền tố: ngắn hơn đứng sau)
        return (0, 0, neg, ident)

    return sorted(logs, key=key)


def reorder_desc_oracle(logs):
    letters = [l for l in logs if not l.split(" ", 1)[1][0].isdigit()]
    digits = [l for l in logs if l.split(" ", 1)[1][0].isdigit()]
    out = []
    while letters:
        best = 0
        for i in range(1, len(letters)):
            ci, ii = letters[i].split(" ", 1)[1], letters[i].split(" ", 1)[0]
            cb, ib = letters[best].split(" ", 1)[1], letters[best].split(" ", 1)[0]
            if ci > cb or (ci == cb and ii < ib):            # chọn "lớn nhất" theo nội dung, rồi định danh nhỏ nhất
                best = i
        out.append(letters.pop(best))
    return out + digits


def reorder_digit_sum(logs):
    """Biến thể 4: letter-log như đề; digit-log sắp theo tổng các chữ số tăng dần (ổn định)."""
    def key(log):
        ident, content = log.split(" ", 1)
        if content[0].isdigit():
            return (1, sum(int(x) for x in content.split(" ")))
        return (0, content, ident)

    return sorted(logs, key=key)


def argsort_logs(logs):
    """Biến thể 5: chỉ số gốc của các log theo thứ tự đầu ra (ổn định)."""
    def key(i):
        ident, content = logs[i].split(" ", 1)
        return (1,) if content[0].isdigit() else (0, content, ident)

    return sorted(range(len(logs)), key=key)


rng = random.Random(15)
for _ in range(500):
    logs = [rand_log(rng) for _ in range(rng.randint(1, 8))]
    assert reorder_desc(logs) == reorder_desc_oracle(logs), logs
    ds = reorder_digit_sum(logs)
    digs = [l for l in ds if l.split(" ", 1)[1][0].isdigit()]
    sums = [sum(int(x) for x in l.split(" ", 1)[1].split(" ")) for l in digs]
    assert sums == sorted(sums)
    assert [logs[i] for i in argsort_logs(logs)] == Solution().reorderLogFiles(logs)
print("giảm dần, digit theo tổng chữ số, argsort: khớp oracle/tính chất trên 500 dãy log")

giảm dần, digit theo tổng chữ số, argsort: khớp oracle/tính chất trên 500 dãy log


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tách nhóm | hai danh sách, sort letter | `O(n log n · L)` | `O(n·L)` | trực quan |
| 2. `sorted` khoá tuple | `(0,c,i)` / `(1,)` | `O(n log n · L)` | `O(n·L)` | **đáp án chính** |
| 3. `cmp_to_key` | bộ so sánh tuỳ biến | `O(n log n · L)` | `O(n·L)` | luật phức tạp |
| 4. Sắp chèn ổn định | tự viết | `O(n² · L)` | `O(n·L)` | không dùng `sort` |

Cả bốn khớp oracle chọn-cực-tiểu trên 1 956 dãy log vét cạn và hàng nghìn dãy ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().reorderLogFiles(["a1 9 2", "b1 hello world", "c1 hello", "a0 3 4"]))
print(Solution().reorderLogFiles(["x2 foo", "x1 foo", "y 7"]))

['c1 hello', 'b1 hello world', 'a1 9 2', 'a0 3 4']
['x1 foo', 'x2 foo', 'y 7']


### Bài 2 (tính tay)

`logs = ["m1 car", "k9 7 7", "m0 car", "a5 bus stop", "b2 8", "a4 bus"]`. (a) Phân loại từng log. (b) Thứ tự letter-log? (c) Kết quả cuối? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện khoá tuple:

In [30]:
def key_todo(log):
    ident, content = log.split(" ", 1)
    pass                # TODO: digit-log trả (1,), letter-log trả (0, content, ident)


# Khi làm xong, bỏ comment để kiểm tra:
# assert [sorted(l, key=key_todo) for l, _ in examples] == [w for _, w in examples]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **`cmp_to_key`**:

In [31]:
class MySolution:
    def reorderLogFiles(self, logs):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().reorderLogFiles(list(l)) == w for l, w in examples)

### Bài 5 (mini challenge): Sort the People / Sort Array by Increasing Frequency (1636)

Viết `frequency_sort(nums)`: sắp tăng theo **tần suất**, cùng tần suất thì sắp **giảm dần theo giá trị** (khoá tuple `(tần suất, −giá trị)`). Kiểm tra bằng oracle chọn cực tiểu.

In [32]:
def frequency_sort(nums):
    pass            # TODO


# Khi làm xong:
# assert frequency_sort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2] and frequency_sort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: letter-log trước; sắp theo nội dung rồi định danh.
- Mức 2: letter: `b1 hello world`, `c1 hello`; nội dung `"hello"` là tiền tố của `"hello world"` nên `c1 hello` trước.
- Mức 3: `['c1 hello', 'b1 hello world', 'a1 9 2', 'a0 3 4']` và `['x1 foo', 'x2 foo', 'y 7']` (nội dung trùng ⇒ định danh `x1 < x2`).

**Bài 2**

- Mức 1: letter: `m1 car`, `m0 car`, `a5 bus stop`, `a4 bus`; digit: `k9 7 7`, `b2 8`.
- Mức 2: nội dung: `bus` < `bus stop` < `car`; `car` trùng: `m0` < `m1`.
- Mức 3: `['a4 bus', 'a5 bus stop', 'm0 car', 'm1 car', 'k9 7 7', 'b2 8']`.

**Bài 3**

- Mức 1: `content[0].isdigit()`.
- Mức 2: digit ⇒ `(1,)`.
- Mức 3: letter ⇒ `(0, content, ident)`.

**Bài 4**

- Mức 1: `cmp(a, b)` trả `-1/0/1`.
- Mức 2: hai digit ⇒ `0`; letter trước digit.
- Mức 3: hai letter ⇒ so `(nội dung, định danh)`.

**Bài 5**

- Mức 1: `Counter` đếm tần suất.
- Mức 2: khoá `(cnt[x], -x)`.
- Mức 3: `sorted(nums, key=lambda x: (cnt[x], -x))`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
a = Solution().reorderLogFiles(["a1 9 2", "b1 hello world", "c1 hello", "a0 3 4"])
b = Solution().reorderLogFiles(["x2 foo", "x1 foo", "y 7"])
assert a == ["c1 hello", "b1 hello world", "a1 9 2", "a0 3 4"] and b == ["x1 foo", "x2 foo", "y 7"]
assert a == oracle(["a1 9 2", "b1 hello world", "c1 hello", "a0 3 4"])
print("Bài 1 OK:", a, b)

Bài 1 OK: ['c1 hello', 'b1 hello world', 'a1 9 2', 'a0 3 4'] ['x1 foo', 'x2 foo', 'y 7']


### Bài 2

Kết quả: `['a4 bus', 'a5 bus stop', 'm0 car', 'm1 car', 'k9 7 7', 'b2 8']`.

In [34]:
logs = ["m1 car", "k9 7 7", "m0 car", "a5 bus stop", "b2 8", "a4 bus"]
want = ["a4 bus", "a5 bus stop", "m0 car", "m1 car", "k9 7 7", "b2 8"]
assert Solution().reorderLogFiles(logs) == want == oracle(logs)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def key_answer(log):
    ident, content = log.split(" ", 1)
    return (1,) if content[0].isdigit() else (0, content, ident)


assert [sorted(l, key=key_answer) for l, _ in examples] == [w for _, w in examples]
for l in tests[:300]:
    assert sorted(l, key=key_answer) == oracle(l)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def reorderLogFiles(self, logs):
        def parts(log):
            ident, content = log.split(" ", 1)
            return ident, content, content[0].isdigit()

        def cmp(a, b):
            ia, ca, da = parts(a)
            ib, cb, db = parts(b)
            if da and db:
                return 0
            if da:
                return 1
            if db:
                return -1
            return -1 if (ca, ia) < (cb, ib) else (1 if (ca, ia) > (cb, ib) else 0)

        return sorted(logs, key=cmp_to_key(cmp))


assert all(MySolutionAnswer().reorderLogFiles(list(l)) == w for l, w in examples)
for l in tests[:300]:
    assert MySolutionAnswer().reorderLogFiles(list(l)) == oracle(l)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
from collections import Counter


def frequency_sort_answer(nums):
    cnt = Counter(nums)
    return sorted(nums, key=lambda x: (cnt[x], -x))


def frequency_sort_oracle(nums):
    pool, out = list(nums), []
    cnt = Counter(nums)
    while pool:
        best = 0
        for i in range(1, len(pool)):
            if (cnt[pool[i]], -pool[i]) < (cnt[pool[best]], -pool[best]):
                best = i
        out.append(pool.pop(best))
    return out


assert frequency_sort_answer([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2] and frequency_sort_answer([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
rng = random.Random(5)
for _ in range(300):
    a = [rng.randint(-3, 3) for _ in range(rng.randint(1, 10))]
    assert frequency_sort_answer(a) == frequency_sort_oracle(a)
print("Bài 5 OK: khớp oracle chọn cực tiểu")

Bài 5 OK: khớp oracle chọn cực tiểu


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (sắp xếp nhiều tiêu chí, một nhóm giữ thứ tự → khoá tuple + sort ổn định):**

```python
class Solution:
    def reorderLogFiles(self, logs: List[str]) -> List[str]:
        def key(log):
            ident, content = log.split(" ", 1)
            return (1,) if content[0].isdigit() else (0, content, ident)
        return sorted(logs, key=key)
```

**Checklist trước khi submit:**

- [ ] Tách bằng `split(" ", 1)` (giữ dấu cách trong nội dung)?
- [ ] Phân loại bằng `content[0].isdigit()` (không `content.isdigit()`)?
- [ ] Khoá letter-log `(0, nội dung, định danh)`; digit-log `(1,)` (bằng nhau ⇒ giữ thứ tự nhờ sort ổn định)?
- [ ] Letter-log **trước** digit-log?
- [ ] Nội dung trùng thì so **định danh**?
- [ ] Đã thử 2 ví dụ, toàn digit, nội dung trùng, nội dung là tiền tố?
- [ ] Đúng tên `class Solution`, `reorderLogFiles(self, logs)`, không in debug.